# 🌲 Forest Inspection Dataset - Explorer

Khám phá dataset: xem ảnh mẫu, thống kê class, phân tích phân bố.

**Dataset**: [Zenodo 15511426](https://zenodo.org/records/15511426) | **Paper**: [arXiv:2403.06621](https://arxiv.org/abs/2403.06621)

In [ ]:
import sys
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Adjust path for Kaggle
# DATA_ROOT = Path('/kaggle/input/forest-sunny')  # Kaggle
DATA_ROOT = Path('data/forest_sunny')  # Local

sys.path.insert(0, '.')
from src.data.dataset import LABEL_COLORS, CLASS_NAMES, rgb_to_class_id, class_id_to_rgb

## 1. Dataset Overview

In [ ]:
# Count images per sequence
print('📊 Dataset Overview')
print('=' * 50)
total = 0
for seq_dir in sorted(DATA_ROOT.iterdir()):
    if seq_dir.is_dir() and seq_dir.name.startswith('seq'):
        color_dir = seq_dir / 'color'
        n = len(list(color_dir.glob('*.png'))) if color_dir.exists() else 0
        total += n
        print(f'  {seq_dir.name}: {n:,} images')
print(f'  Total: {total:,} image pairs')

## 2. Sample Visualization

In [ ]:
# Show samples from each altitude
sample_seqs = ['seq1', 'seq5', 'seq9']  # 30m/0°, 50m/-60°, 80m/-90°

fig, axes = plt.subplots(len(sample_seqs), 2, figsize=(14, 5*len(sample_seqs)))

for i, seq in enumerate(sample_seqs):
    color_dir = DATA_ROOT / seq / 'color'
    label_dir = DATA_ROOT / seq / 'labels'
    files = sorted(color_dir.glob('*.png'))
    mid = len(files) // 2  # Take middle image
    
    img = Image.open(files[mid])
    lbl = Image.open(label_dir / files[mid].name)
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{seq} - RGB', fontsize=12)
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(lbl)
    axes[i, 1].set_title(f'{seq} - Labels', fontsize=12)
    axes[i, 1].axis('off')

# Legend
patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in zip(CLASS_NAMES, LABEL_COLORS)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=9)
plt.tight_layout()
plt.show()

## 3. Class Distribution Analysis

In [ ]:
# Analyze class pixel distribution (sampled)
from collections import Counter

def analyze_distribution(seq_name, n_samples=30):
    label_dir = DATA_ROOT / seq_name / 'labels'
    files = sorted(label_dir.glob('*.png'))
    step = max(1, len(files) // n_samples)
    sampled = files[::step][:n_samples]
    
    total = 0
    counts = Counter()
    for f in sampled:
        arr = np.array(Image.open(f).convert('RGB'))
        total += arr.shape[0] * arr.shape[1]
        for cid, color in enumerate(LABEL_COLORS):
            mask = np.all(arr == np.array(color), axis=-1)
            counts[CLASS_NAMES[cid]] += mask.sum()
    
    return {k: v/total*100 for k, v in counts.items()}

# Analyze 3 sequences
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, seq in zip(axes, ['seq1', 'seq5', 'seq9']):
    dist = analyze_distribution(seq)
    sorted_dist = dict(sorted(dist.items(), key=lambda x: -x[1]))
    colors = [np.array(LABEL_COLORS[CLASS_NAMES.index(n)])/255 for n in sorted_dist]
    ax.barh(list(sorted_dist.keys()), list(sorted_dist.values()), color=colors)
    ax.set_title(f'{seq} Class Distribution', fontsize=11)
    ax.set_xlabel('Percentage (%)')

plt.tight_layout()
plt.show()

## 4. Camera Metadata

In [ ]:
import pandas as pd

# Read airsim_rec.txt
meta = pd.read_csv(
    DATA_ROOT / 'seq1' / 'airsim_rec.txt',
    sep='\t',
    names=['ImageFile', 'TimeStamp', 'POS_X', 'POS_Y', 'POS_Z', 'Q_W', 'Q_X', 'Q_Y', 'Q_Z']
)
print(f'seq1 metadata: {len(meta)} entries')
print(meta.head())

# Plot flight path
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(meta['POS_X'], meta['POS_Y'], 'b-', alpha=0.5, linewidth=0.5)
ax.scatter(meta['POS_X'].iloc[0], meta['POS_Y'].iloc[0], c='green', s=100, label='Start', zorder=5)
ax.scatter(meta['POS_X'].iloc[-1], meta['POS_Y'].iloc[-1], c='red', s=100, label='End', zorder=5)
ax.set_title('seq1 - Drone Flight Path (Top View)', fontsize=12)
ax.set_xlabel('X position')
ax.set_ylabel('Y position')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()